# TensionMagnetique — `tension_magnetique`

`Postraitement_fields` · ASN 28 validation form (TRUST-LiMe).



## Introduction

Validation of the **magnetic tension** post-treatment field of the standalone
magnetic (induction) equation:

$$\partial_t b + (u \cdot \nabla) b - \nabla \cdot (\eta \nabla b) = 0$$

This notebook runs **one and only one test case**: `tension_magnetique`. Per the
TRUST validation-form layout the other `Postraitement_fields` cases live in
their own sibling folders.

## What it validates

$$\mathbf{F}_t = \frac{1}{\mu_0}(\mathbf{b}\cdot\nabla)\mathbf{b},\qquad
(\mathbf{F}_t)_i = \frac{1}{\mu_0}\sum_j b_j\,\frac{\partial b_i}{\partial x_j}$$

The per-element gradient $\nabla b$ is reconstructed by **Green-Gauss**
(the kernel has no element-field gradient factory on PolyMAC: `Discret_Thyd::grad_T`
is a stub on this discretisation and the generic `Gradient` post operator rejects
PolyMAC element fields), so the module computes it itself from the P0 unknown.

## Contract checked

| # | Quantity | Formula |
|---|---|---|
| 1 | magnetic tension force density | $F_t = \mu_0^{-1}(b\cdot\nabla)b$ |

Vector model (`nb_composantes 2`): the scalar model is rejected with a clean
error by `creer_champ` (`(b\cdot\nabla)b` is only meaningful for a full vector b).



In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()



## Helpers: read a TRUST probe (`.son`) file

The case is copied by `trustutils` into `run.BUILD_DIRECTORY`; the `.son`
probe files are written next to the dataset in `src/` and copied over in the
run cell below, where `read_son()` expects them.



In [ ]:
import numpy as np
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, sonde):
    """Return (time, values) for probe `sonde` of case `case`."""
    path = f"{BUILD}/{case}_{sonde}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def read_son_all(case, sonde):
    """Like read_son but returns every entry (components of a vector field)."""
    path = f"{BUILD}/{case}_{sonde}.son"
    son = SonFile(path, None)
    return [(e, np.asarray(son.getValues(e)[1], dtype=float)) for e in son.getEntries()]

print("BUILD =", BUILD)



## Run

Sequential-only: the dataset is run by `trustutils` on one processor
(no `partition()`, no `nbProcs`).



In [ ]:
import pathlib, shutil

from trustutils import run

run.reset()
run.addCase(".", "tension_magnetique.data")      # sequential-only: never nbProcs > 1
run.printCases()
run.runCases()

# trustutils runs the case in its build tree; the .son probes are written next
# to the dataset in src/, so copy them where read_son() looks for them.
build = pathlib.Path(run.BUILD_DIRECTORY)
build.mkdir(parents=True, exist_ok=True)
for son in pathlib.Path("src").glob("tension_magnetique_*.son"):
    shutil.copy2(son, build / son.name)



## Checks


In [ ]:
entries = read_son_all("tension_magnetique", "SONDE_FT")
for name, v in entries:
    print(name, "=", v[-1])
comps = [v[-1] for name, v in entries]

# b = (y, x) is exactly linear => grad b = [[0,1],[1,0]] at every cell centre,
# so Ft = (b . grad) b / mu0 = (x, y) with mu0 = 1.
expected = np.array([0.45, 0.55])
print(f"tension_magnetique (probe) : ({comps[0]:.6g}, {comps[1]:.6g})  (expected {expected})")
err = np.max(np.abs(np.array(comps) - expected)) / np.max(np.abs(expected))
print(f"relative error             : {err:.3e}")
assert err < 1e-12, f"tension_magnetique wrong: {comps}"
print("OK: tension_magnetique = (b . grad) b / mu0.")



## Computer Performance


In [ ]:
run.tablePerf()

